<a href="https://colab.research.google.com/github/vipulsahu0629/PW_Assessment/blob/main/Mongoose_and_Express.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mongoose and Express | Assignment
**Assignment Code:** FSD-AG-014                     
 **Total Marks:** 220

## **Question 1)** List two benefits of using Mongoose instead of the native MongoDB driver. Also mention one scenario where using the native driver could be preferable.

**Answer:**

**Two benefits of Mongoose:**

1. **Schema validation:** Mongoose lets you define a schema (required fields, types, min/max length, custom validators, etc.), so invalid data is rejected automatically before it ever reaches the database — the native driver has no built-in validation, so you'd have to write all of that by hand.
2. **Convenient modeling & middleware:** Mongoose models give you built-in methods (`.find()`, `.save()`, `.updateOne()`, virtuals, instance/static methods) plus lifecycle hooks (`pre`/`post` middleware, e.g. hashing a password before saving), which makes application code cleaner and more expressive than writing raw driver calls everywhere.

**When the native driver could be preferable:**

For **performance-critical, high-throughput operations** (e.g. bulk inserts of millions of documents, or advanced aggregation pipelines) where Mongoose's extra validation/casting layer adds overhead you don't need, the native `mongodb` driver gives you closer-to-the-metal performance and finer control over exactly what's sent to MongoDB.

## **Question 2)** Create a Mongoose schema for a `User` with properties:
- `name` (required string, minimum length 2, maximum length 50)
- `email` (required string)

Then export a `User` model.

**Answer:**

```js
// models/User.js
const mongoose = require('mongoose');

const userSchema = new mongoose.Schema({
  name: {
    type: String,
    required: true,
    minlength: 2,
    maxlength: 50,
  },
  email: {
    type: String,
    required: true,
  },
});

const User = mongoose.model('User', userSchema);

module.exports = User;
```

`new mongoose.Schema({...})` defines the shape and validation rules for documents in the `users` collection. `mongoose.model('User', userSchema)` compiles the schema into a usable model, which is then exported for use elsewhere in the app.

## **Question 3)** Write code to (a) fetch all users, (b) fetch a user by ID, and (c) add a query helper `byEmailDomain(domain)` that returns users whose email ends with the given domain. Show example usage.

**Answer:**

```js
// models/User.js
const mongoose = require('mongoose');

const userSchema = new mongoose.Schema({
  name: { type: String, required: true, minlength: 2, maxlength: 50 },
  email: { type: String, required: true },
});

// (c) Query helper: filters users whose email ends with the given domain
userSchema.query.byEmailDomain = function (domain) {
  return this.where('email').regex(new RegExp(domain + '$', 'i'));
};

const User = mongoose.model('User', userSchema);
module.exports = User;
```

```js
// userQueries.js
const User = require('./models/User');

// (a) Fetch all users
async function getAllUsers() {
  return User.find();
}

// (b) Fetch a user by ID
async function getUserById(id) {
  return User.findById(id);
}

// Example usage, including the query helper (c)
async function run() {
  const allUsers = await getAllUsers();
  console.log('All users:', allUsers);

  const oneUser = await getUserById('64f1a2b3c4d5e6f7a8b9c0d1');
  console.log('User by ID:', oneUser);

  // (c) Using the byEmailDomain query helper
  const gmailUsers = await User.find().byEmailDomain('@gmail.com');
  console.log('Users with @gmail.com emails:', gmailUsers);
}

run();
```

`User.find()` fetches every document, and `User.findById(id)` fetches a single document matching the given `_id`. The custom **query helper** `byEmailDomain` is attached to `userSchema.query`, letting it be chained onto any query (e.g. `User.find().byEmailDomain(...)`) — internally it just adds a regex filter on `email`.

## **Question 4)** Demonstrate two ways to change a user's name to "Rita" using Mongoose: (1) load, modify, then `save()`, and (2) use `findOneAndUpdate()`. Mention when `save()` is preferable.

**Answer:**

```js
// Method 1: load, modify, then save()
async function updateNameWithSave(userId) {
  const user = await User.findById(userId);
  user.name = 'Rita';
  await user.save();
  console.log('Updated (save):', user);
}
```

```js
// Method 2: findOneAndUpdate()
async function updateNameWithFindOneAndUpdate(userId) {
  const updatedUser = await User.findOneAndUpdate(
    { _id: userId },
    { name: 'Rita' },
    { new: true } // return the updated document, not the original
  );
  console.log('Updated (findOneAndUpdate):', updatedUser);
}
```

**When `save()` is preferable:**

`save()` is better when you need to **read the document first** — e.g. to run custom logic based on its current values, trigger Mongoose's `pre('save')` / `post('save')` middleware hooks (like re-hashing a changed password), or run full document validation on the entire document rather than just the updated fields. `findOneAndUpdate()` is faster for simple, direct field updates since it skips loading the full document into memory, but it bypasses some Mongoose document middleware by default.

## **Question 5)** Explain when you would embed a subdocument versus using references in MongoDB/Mongoose. Show code for a one-to-many reference (e.g. a `Post` having many `Comment` references).

**Answer:**

**Embed a subdocument when:**
- The related data is small, doesn't change often, and is almost always accessed **together** with the parent (e.g. an `address` embedded inside a `User`).
- You want to fetch everything in a single query, with no extra `$lookup`/join needed.

**Use references when:**
- The related data can grow large or unbounded (e.g. a post could have thousands of comments — embedding all of them would make the document huge and slow to load).
- The related documents need to be queried, updated, or displayed **independently** of the parent.
- Multiple documents need to reference the same related document (avoiding duplication).

**One-to-many reference example (Post → many Comments):**

```js
// models/Post.js
const mongoose = require('mongoose');

const postSchema = new mongoose.Schema({
  title: { type: String, required: true },
  body: { type: String, required: true },
});

module.exports = mongoose.model('Post', postSchema);
```

```js
// models/Comment.js
const mongoose = require('mongoose');

const commentSchema = new mongoose.Schema({
  text: { type: String, required: true },
  post: { type: mongoose.Schema.Types.ObjectId, ref: 'Post', required: true },
});

module.exports = mongoose.model('Comment', commentSchema);
```

```js
// Fetching a post along with its comments
const Post = require('./models/Post');
const Comment = require('./models/Comment');

async function getPostWithComments(postId) {
  const post = await Post.findById(postId);
  const comments = await Comment.find({ post: postId });
  return { post, comments };
}
```

Each `Comment` stores a `post` field of type `ObjectId` (with `ref: 'Post'`) pointing back to its parent post — this is the "many" side of the one-to-many relationship, and `ref` also enables Mongoose's `.populate()` for fetching the referenced document automatically if needed.

## **Question 6)** Write a Mongoose (or MongoDB) aggregation pipeline to group users by email domain and count how many users per domain. Then show how you could use `$lookup` to join a `bounces` collection (with bounce counts per domain).

**Answer:**

```js
// Group users by email domain and count them
async function usersByDomain() {
  const result = await User.aggregate([
    {
      $project: {
        domain: { $arrayElemAt: [{ $split: ['$email', '@'] }, 1] },
      },
    },
    {
      $group: {
        _id: '$domain',
        userCount: { $sum: 1 },
      },
    },
  ]);
  console.log(result);
  // e.g. [ { _id: 'gmail.com', userCount: 12 }, { _id: 'yahoo.com', userCount: 5 } ]
}
```

**Joining with a `bounces` collection using `$lookup`:**

```js
async function usersByDomainWithBounces() {
  const result = await User.aggregate([
    {
      $project: {
        domain: { $arrayElemAt: [{ $split: ['$email', '@'] }, 1] },
      },
    },
    {
      $group: {
        _id: '$domain',
        userCount: { $sum: 1 },
      },
    },
    {
      $lookup: {
        from: 'bounces',        // the bounces collection
        localField: '_id',      // domain from the grouped result
        foreignField: 'domain', // matching field in bounces
        as: 'bounceInfo',
      },
    },
  ]);
  console.log(result);
});
```

**Explanation:**
- `$project` extracts the domain from each user's email by splitting on `@` and taking the second part.
- `$group` groups documents by `domain` and counts users per group with `$sum: 1`.
- `$lookup` performs a **left outer join** against the `bounces` collection, matching each domain (`_id` from the group stage) to `bounces.domain`, attaching the matching bounce documents as a `bounceInfo` array on each result.

## **Question 7)** Show minimal code (ESM) to connect to MongoDB Atlas using Mongoose. Assume the URI is stored in the environment variable `MONGODB_URI`.

**Answer:**

```js
// db.mjs
import mongoose from 'mongoose';

async function connectDB() {
  try {
    await mongoose.connect(process.env.MONGODB_URI);
    console.log('Connected to MongoDB Atlas');
  } catch (error) {
    console.error('MongoDB connection error:', error.message);
    process.exit(1);
  }
}

export default connectDB;
```

```js
// index.mjs
import connectDB from './db.mjs';

connectDB();
```

Since this uses **ESM** (`import`/`export`) instead of CommonJS, the file extension is `.mjs` (or `"type": "module"` set in `package.json`). `mongoose.connect(uri)` opens the connection to the Atlas cluster using the connection string stored securely in the `MONGODB_URI` environment variable, rather than hardcoding credentials in the source code.

## **Question 8)** Give a short description of Express. Then write routes for:
- `GET /health` → responds "OK"
- `GET /users/:id` → responds with `{ id: <id> }`
- `GET /search?term=...` → responds with `{ term: <term> }`

**Answer:**

**Express** is a minimal, flexible web framework for Node.js. It provides a simple API for defining routes, handling HTTP requests/responses, and composing **middleware** (functions that run in sequence for each request), making it much easier to build web servers and REST APIs than using the raw `http` module directly.

```js
// app.js
const express = require('express');
const app = express();

app.get('/health', (req, res) => {
  res.send('OK');
});

app.get('/users/:id', (req, res) => {
  res.json({ id: req.params.id });
});

app.get('/search', (req, res) => {
  res.json({ term: req.query.term });
});

app.listen(3000, () => {
  console.log('Server listening on http://localhost:3000');
});
```

`req.params.id` reads the `:id` URL parameter, while `req.query.term` reads the `term` value from the query string (e.g. `?term=shoes`).

## **Question 9)** Create a router for `/api/users` with two endpoints: `GET /` returning empty array and `POST /` returning the posted JSON. Also add a logger middleware that prints method and URL for every request.

**Answer:**

```js
// routes/users.js
const express = require('express');
const router = express.Router();

router.get('/', (req, res) => {
  res.json([]);
});

router.post('/', (req, res) => {
  res.json(req.body);
});

module.exports = router;
```

```js
// app.js
const express = require('express');
const usersRouter = require('./routes/users');

const app = express();

app.use(express.json()); // parses incoming JSON request bodies

// Logger middleware: runs for every request
app.use((req, res, next) => {
  console.log(`${req.method} ${req.url}`);
  next();
});

app.use('/api/users', usersRouter);

app.listen(3000, () => {
  console.log('Server listening on http://localhost:3000');
});
```

`express.Router()` creates a mini, mountable router — `app.use('/api/users', usersRouter)` prefixes every route defined in `usersRouter` with `/api/users`. The logger middleware is registered with `app.use()` **before** the router, so it runs on every incoming request; calling `next()` passes control on to the next matching handler.

## **Question 10)** Add an error-handling middleware to catch thrown errors and respond with `{ error: <message> }`, status 500. Demonstrate by a route that throws an error.

**Answer:**

```js
// app.js
const express = require('express');
const app = express();

// A route that throws an error
app.get('/boom', (req, res) => {
  throw new Error('Something went wrong!');
});

// Error-handling middleware (must have 4 arguments)
app.use((err, req, res, next) => {
  console.error(err.stack);
  res.status(500).json({ error: err.message });
});

app.listen(3000, () => {
  console.log('Server listening on http://localhost:3000');
});
```

Express recognizes a middleware as an **error handler** specifically because it takes **four** arguments `(err, req, res, next)`. When a synchronous route handler throws (or `next(err)` is called explicitly for async errors), Express skips all normal middleware/routes and jumps straight to this error handler, which responds with a `500` status and a JSON error message instead of crashing the server.

## **Question 11)** Write middleware `auth()` that checks for `Authorization: Bearer <token>` header, verifies the token, and sets `req.user`. Then protect route `GET /me` to return user info if valid, otherwise 401.

**Answer:**

```js
// middleware/auth.js
const jwt = require('jsonwebtoken');

function auth(req, res, next) {
  const authHeader = req.headers['authorization'];

  if (!authHeader || !authHeader.startsWith('Bearer ')) {
    return res.status(401).json({ error: 'No token provided' });
  }

  const token = authHeader.split(' ')[1];

  try {
    const decoded = jwt.verify(token, process.env.JWT_SECRET);
    req.user = decoded; // attach the decoded user payload to the request
    next();
  } catch (error) {
    return res.status(401).json({ error: 'Invalid or expired token' });
  }
}

module.exports = auth;
```

```js
// app.js
const express = require('express');
const auth = require('./middleware/auth');

const app = express();

app.get('/me', auth, (req, res) => {
  res.json({ user: req.user });
});

app.listen(3000, () => {
  console.log('Server listening on http://localhost:3000');
});
```

**Explanation:**
- `req.headers['authorization']` reads the header; if it's missing or doesn't start with `"Bearer "`, the request is rejected with `401` immediately.
- `jwt.verify(token, secret)` checks the token's signature and expiry; if valid, it returns the decoded payload, which is attached to `req.user` so later handlers can use it.
- Passing `auth` as a second argument to `app.get('/me', auth, handler)` makes it run **before** the route handler — only requests with a valid token reach the actual `/me` logic; everything else gets a `401` from inside the middleware.